In [1]:
import numpy as np
import optimistix as optx
from atmodeller import (
    EquilibriumModel,
    Planet,
    ChemicalSpecies,
    SpeciesNetwork,
    debug_logger,
    earth_oceans_to_hydrogen_mass,
    SolverParameters,
)
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer
from atmodeller.interfaces import FugacityConstraintProtocol, ThermodynamicStateProtocol
from atmodeller import debug_logger
from atmodeller.classes import EquilibriumModel
from atmodeller.containers import (
    ChemicalSpecies,
    FixedFugacityConstraint,
    SpeciesNetwork,
    ThermodynamicState,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

from scipy.interpolate import interp1d
from scipy.optimize import fsolve, root_scalar
import os

from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
formatter.set_useOffset(False)
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

logger = debug_logger()

Atmodeller initialized with double precision (float64)


In [2]:
color_tot = "black"
color_H2 = "orange"
color_H2O = "blue"
color_SiH4 = "red"
color_SiO2 = "yellow"
color_SiO = "brown"
color_O2 = "limegreen"
color_CO2 = "cyan"
color_CO = "magenta"
color_CH4 = "purple"
color_N2 = "green"
color_NH3 = "pink"
color_MgO = "darkred"
color_He = "gray"
color_H = "orange"
color_C = "cyan"
color_N = "green"
color_O = "blue"
color_Si = "red"
color_Mg = "lime"
color_MgH2 = "gold"
color_MgH = "teal"
color_MgOH = "indigo"
color_Ccr = "dimgray"

In [3]:
SiO2_mantlemasspercent_palme14: float = 45.4
total_mantlemasspercent_palme14: float = 98.41
core_mass_fraction: float = 0.327  # typical values used are between 0.325 - 0.33
Si_massfraction_palme14: float = round(
    SiO2_mantlemasspercent_palme14
    / total_mantlemasspercent_palme14
    * (1 - core_mass_fraction)
    / (28.0855 + 2 * 15.999)
    * 28.0855,
    3,
)

O_massfraction_palme14: float = Si_massfraction_palme14 / 28.0855 * 2 * 15.999
# Lodders et al. (2009) Springer Book Chapter - Table 8 (in wt%)

H_masspercent_lodders09: float = 73.9
He_masspercent_lodders09: float = 24.69
C_masspercent_lodders09: float = 0.22
N_masspercent_lodders09: float = 0.07
O_masspercent_lodders09: float = 0.63
Si_masspercent_lodders09: float = 0.07
Mg_masspercent_lodders09: float = 0.06

# Lodders et al. (2009) Springer Book Chapter - Table 6 (log-normalalized abundances relative to H)

H_logN: float = 12
He_logN: float = 10.93
C_logN: float = 8.39
N_logN: float = 7.86
O_logN: float = 8.73
Si_logN: float = 7.53
Mg_logN: float = 7.54 

In [4]:
eos_models = get_eos_models()
sol_models = get_solubility_models()

H2O_g = ChemicalSpecies.create_gas("H2O")
H2O_gs = ChemicalSpecies.create_gas("H2O", solubility=sol_models["H2O_peridotite_sossi23"])
H2O_rgs = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
    solubility=sol_models["H2O_peridotite_sossi23"],
)
H2O_rg = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
)

H2_g = ChemicalSpecies.create_gas("H2")
H2_gs = ChemicalSpecies.create_gas("H2", solubility=sol_models["H2_basalt_hirschmann12"])
H2_rgs = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"], solubility=sol_models["H2_basalt_hirschmann12"]
)
H2_rg = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"]
)

O2_g = ChemicalSpecies.create_gas("O2")
O2_rg = ChemicalSpecies.create_gas("O2", activity=eos_models["O2_cs_shi92"])

OSi_g = ChemicalSpecies.create_gas("OSi")
OSi_rg = ChemicalSpecies.create_gas("OSi", activity=eos_models["OSi_rk49_connolly16"])

H4Si_g = ChemicalSpecies.create_gas("H4Si")
H4Si_rg = ChemicalSpecies.create_gas("H4Si", activity=eos_models["H4Si_wang18"])

O2Si_cd = ChemicalSpecies.create_condensed("O2Si", state="cd")

C_cd = ChemicalSpecies.create_condensed("C", state="cd")
CSi_cd = ChemicalSpecies.create_condensed("CSi", state="cd")
Si_cd = ChemicalSpecies.create_condensed("Si", state="cd")
N4Si3_cd = ChemicalSpecies.create_condensed("N4Si3", state="cd")
H2O_cd = ChemicalSpecies.create_condensed("H2O", state="cd")

CO2_g = ChemicalSpecies.create_gas("CO2")
CO2_gs = ChemicalSpecies.create_gas("CO2", solubility=sol_models["CO2_basalt_dixon95"])
CO2_rgs = ChemicalSpecies.create_gas(
    "CO2", solubility=sol_models["CO2_basalt_dixon95"], activity=eos_models["CO2_cs_shi92"]
)
CO2_rg = ChemicalSpecies.create_gas(
    "CO2",activity=eos_models["CO2_cs_shi92"]
)

CO_g = ChemicalSpecies.create_gas("CO")
CO_gs = ChemicalSpecies.create_gas("CO", solubility=sol_models["CO_basalt_yoshioka19"])
CO_rgs = ChemicalSpecies.create_gas(
    "CO", solubility=sol_models["CO_basalt_yoshioka19"], activity=eos_models["CO_cs_shi92"]
)
CO_rg = ChemicalSpecies.create_gas(
    "CO", activity=eos_models["CO_cs_shi92"]
)

CH4_g = ChemicalSpecies.create_gas("CH4")
CH4_gs = ChemicalSpecies.create_gas("CH4", solubility=sol_models["CH4_basalt_ardia13"])
CH4_rgs = ChemicalSpecies.create_gas(
    "CH4", solubility=sol_models["CH4_basalt_ardia13"], activity=eos_models["CH4_cs_shi92"]
)
CH4_rg = ChemicalSpecies.create_gas(
    "CH4", activity=eos_models["CH4_cs_shi92"]
)

N2_g = ChemicalSpecies.create_gas("N2")
N2_gs = ChemicalSpecies.create_gas("N2", solubility=sol_models["N2_basalt_libourel03"])
N2_rgs = ChemicalSpecies.create_gas(
    "N2",solubility=sol_models["N2_basalt_libourel03"], activity=eos_models["N2_cs_saxena87"]
)
N2_rg = ChemicalSpecies.create_gas(
    "N2", activity=eos_models["N2_cs_saxena87"]
)


NH3_g = ChemicalSpecies.create_gas("H3N")
NH3_rg = ChemicalSpecies.create_gas("H3N", activity=eos_models["H3N_rk49_reid87"])

He_g = ChemicalSpecies.create_gas("He")
He_gs = ChemicalSpecies.create_gas("He", solubility=sol_models["He_basalt_jambon86"])
He_rgs = ChemicalSpecies.create_gas(
    "He", solubility=sol_models["He_basalt_jambon86"], activity=eos_models["He_chabrier21"]
)
He_rg = ChemicalSpecies.create_gas(
    "He", activity=eos_models["He_chabrier21"]
)


species_HOSi_magma_nosol_ideal = SpeciesNetwork((H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_ideal = SpeciesNetwork((H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_real = SpeciesNetwork((H2O_rgs, H2_rgs, O2_rg, OSi_rg, H4Si_rg, O2Si_cd))
species_HOSi_atm_sol_real_gas = SpeciesNetwork((H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg, O2Si_cd))
species_HOSi_atm_sol_real_condensed = SpeciesNetwork((H2O_rg, H2_rg, O2_rg, H2O_cd,OSi_rg, H4Si_rg, O2Si_cd))
#Add SiO2 condensed again, add

species_HHeCNOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)
species_HHeCNOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_gs, CO_gs, CH4_gs, N2_gs, NH3_g, He_gs)
)
species_HHeCNOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        CO2_rgs,
        CO_rgs,
        CH4_rgs,
        N2_rgs,
        NH3_rg,
        He_rgs,
    )
)
#,, , N2_rg, NH3_rg,
species_HHeCNOSi_atmosphere_condensed = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,O2Si_cd, H2O_cd,C_cd, CSi_cd, Si_cd,N4Si3_cd,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg)
)
#C_cd, CSi_cd, Si_cd, N4Si3_cd,,N2_rg, NH3_rg , 
species_HHeCNOSi_atmosphere_gas = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,O2Si_cd, C_cd, CSi_cd, Si_cd,N4Si3_cd,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg)
)
species_HHeCNOSi_atmosphere_gas_noSiO2 = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, C_cd, CSi_cd, Si_cd,N4Si3_cd)
)

species_HHeCNOSi_atmosphere_gas_C = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, C_cd)
)

species_HHeCNOSi_atmosphere_gas_CSi  = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, CSi_cd)
)

species_HHeCNOSi_atmosphere_gas_Si = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, Si_cd)
)

species_HHeCNOSi_atmosphere_gas_N4Si3 = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, N4Si3_cd)
)

species_HHeCNOSi_atmosphere_gas_no_condensed = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg)
)

species_HHeCNOSi_atmosphere_gas_SiO2 = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, O2Si_cd)
)

species_HHeCNOSi_atmosphere_gas_SiO2_and_H2O = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg,
     He_rg,N2_rg, NH3_rg, CO2_rg, CO_rg, CH4_rg, O2Si_cd, H2O_cd)
)

In [5]:
number_of_realisations = 50

# Overall structure 
R_earth = 6.371e6
R_uranus = 3.939 * R_earth
M_earth = 5.972e24 #kg
M_uranus = 14.536 * M_earth

#fO2_log10_shifts = [-1]
fO2_log10_shifts = np.linspace(-6, 0, num=30)  # IW-6 to IW
fO2_log10_shifts = [-6,-5,-4,-3,-2,-1,0]
hmps = np.logspace(-1,0, num=50)  # wt% H
h_kgs = hmps / 100 * M_uranus  # kg

si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09


rho_centres = [2500,3000,2000,1500,1000]
rho_centres = [3000]
rho_centre = 3000
print(rho_centres)
R_core = 0.7*R_uranus
Lowerbound_M_core = [
    4*np.pi*(R_core**3)*rho/3
    for rho in rho_centres
]
M_cores = Lowerbound_M_core
print(M_cores)

[3000]
[6.812158460855618e+25]


In [6]:
f = 1/4

T_solar = 5772
R_solar = 6.957e8
d_solar_Uranus = 2876679082000

T_irr = T_solar * np.sqrt(R_solar/d_solar_Uranus)

T_eff = 59.1 #pm 0.3 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
T_eq = 58.2 #pm 1 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
T_int = (T_eff**(4) - T_eq**(4))**(1/4)
#T_int = 120


gamma_1 = 0.01
gamma_2 = 2
gamma_3 = 0.1
gammas = np.logspace(-4,2,7)
gamma = gamma_2
tau = np.logspace(-10,10,1000)

In [7]:
def Temp_profile(tau, T_int, gamma):
    #This function gives back a Temperature, given optical depth, gamma and intrinsic temperature
    #tau : give an optical depth
    #T_int : give the intrinsic temperature
    #T_irr : taken to be constant along this short evolutionary track
    #gamma : gamme based on opacities of visible on thermal
    return (((3*T_int**(4))/4)*((2/3)+tau) + ((3*f*T_irr**(4))/4)*((2/3)+ (1/(gamma*np.sqrt(3))) + ((gamma/(np.sqrt(3)))- (1/(gamma*np.sqrt(3))))*np.exp(-gamma*tau*np.sqrt(3))))**(1/4)  

def find_tau(tau,T_int, gamma, target, initial_guess):
    #This function gives back the optical depth at which a certain temperature can be found, given several parameters
    #tau: space of optical depths to go over
    #T_int : intrinsic temperature
    #gamma : gives the gamma 
    #target: the target temperature
    #intial_guess : your initial guess for the optical depth
    def g(tau):
        return Temp_profile(tau,T_int, gamma) - target
    x_solution = fsolve(g, initial_guess)
    return x_solution[0]

def find_gamma(tau,T_int, gamma, target, initial_guess):
    #This function gives back the gamma at which a certain temperature can be found, given several parameters
    #tau: optical depth at which it is found
    #T_int : intrinsic temperature
    #gamma : gammas to loop over
    #target: the target temperature
    #intial_guess : your initial guess for the optical depth
    def h(gamma):
        return Temp_profile(tau,T_int, gamma) - target
    x_solution = fsolve(h, initial_guess)
    return x_solution[0]
def find_scale(Temperature,T_int,gamma, Pressure, tau, initial_guess):
    #Probably will not be used in the future, finds how optical depth and Pressure are scaled
    #Temperature: gives the temperature at which to calculate the scaling
    #T_int : intrinsic temperature
    #gamma : gamma
    #Pressure: gives the pressure at which to calculate the scaling 
    #denotes the optical depths to loop over 
    #intiial guess for optical depth
    tau_scale = find_tau(tau,T_int,gamma, Temperature, initial_guess)
    scale_factor = Pressure/tau_scale
    return scale_factor

def find_Tint(tau,T_int, gamma, target, initial_guess):
    #This function gives back the T_intr at which a certain temperature can be found, given several parameters
    #tau: optical depth at which it is found
    #T_int : intrinsic temperature to loop over
    #gamma : gamma
    #target: the target temperature
    #intial_guess : your initial guess for the optical depth
    def ze(T_int):
        return Temp_profile(tau,T_int, gamma) - target
    x_solution = fsolve(ze, initial_guess)
    return x_solution[0]
    

def Pressure_Temperatures(T_original, T_int,gamma, tau, Pressure, initial_guess):
    #Gives Pressures/Temperatures profiles given different parameters
    #T_original: Temperature where we know the pressures
    #T_int : the intrinisic temperatures
    #gamma : gamma 
    #tau: optical depths to find over 
    #Pressure: pressure where we know the temperature
    #intial_guess : initial guess for optical depth
    scales = find_scale(T_original, T_int, Pressure, tau, initial_guess)
    Pressures = tau*scales
    Temperatures = Temp_profile(tau, T_int,gamma)
    indices = [i for i, v in enumerate(Temperatures) if 0 < v < 6000]
    return Pressures[indices], Temperatures[indices]

In [8]:
def peridotite_solidus(P_bar): #On the cooling of a deep terrestrial magma ocean
    #P_bar: pressure in bar
    #Returns: solidus temperature in K
    P_Pa = P_bar*10**5
    T_sol = 1661.2*((P_Pa/(1.336*10**9))+1)**(1/7.437)
    return T_sol

def peridotite_liquidus(P_bar):
    #P_bar: pressure in bar
    #Returns: liquidus temperature in K
    P_Pa = P_bar*10**5
    T_liq = 1982.1*((P_Pa/(6.594*10**9))+1)**(1/5.374)
    
    return T_liq

In [9]:
def opacity(P, T, met): 
    #GASEOUS MEAN OPACITIES FOR GIANT PLANET AND ULTRACOOL DWARF ATMOSPHERES OVER A RANGE OF METALLICITIES AND TEMPERATURES
    #P in bar 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    c1 = 10.602
    c2 = 2.882
    c3 = 6.09e-15
    c4 = 2.954
    c5 = -2.526
    c6 = 0.843
    c7 = -5.490
    
    c8low = -14.051
    c9low = 3.055
    c10low = 0.024
    c11low = 1.877
    c12low = -0.445
    c13low = 0.8321
    
    c8high =  82.241
    c9high = -55.456
    c10high = 8.754 
    c11high = 0.7048
    c12high = -0.0414
    c13high = 0.8321

    if P <= 0 or T <= 0:
        return np.inf
    kappa_low_P = c1*np.arctan(np.log10(T)-c2) - (c3*np.exp((np.log10(T)-c5)**2)/(np.log10(P)+c4)) +c6*met +c7
    if T <= 800:
        kappa_high_P = c8low + c9low*np.log10(T)+c10low*(np.log10(T)**2)+ (c11low + c12low*np.log10(T))*np.log10(P) + c13low*met*(0.5 + (np.arctan(5*np.log10(T)-12.5)/np.pi))
    else:
        kappa_high_P = c8high + c9high*np.log10(T)+c10high*(np.log10(T)**2)+ (c11high + c12high*np.log10(T))*np.log10(P) + c13high*met*(0.5 + (np.arctan(5*np.log10(T)-12.5)/np.pi))
    
    kappa_final = 10**(kappa_low_P) + 10**(kappa_high_P)

    return kappa_final

def optical_depth_finder(P, T, met, surf_grav):
    #Function that gives back optical depths
    #P in cgs
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    opt_depth = P*opacity(P, T, met)/surf_grav
    return opt_depth

def optical_depth_finder_given_opacity(P, opacity, surf_grav):
    #Function that gives back optical depths
    #P in cgs
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    opt_depth = P*opacity/surf_grav
    return opt_depth

def pressure_finder(optical_depths, T, met, surf_grav, initial_guess):
    #optical depths 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    #gives back pressure in dyne/cm^2
    def az(press):
        return optical_depth_finder(press, T, met, surf_grav) - optical_depths
    x_solution = fsolve(az, initial_guess)
    return x_solution[0]

def pressure_finder_given_opacity(optical_depths, opacity, surf_grav):
    #optical depths 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    #gives back pressure in dyne/cm^2
    pressi = optical_depths*surf_grav/opacity
    return pressi

In [10]:
def Profile_From_1_bar(Temp_at_1bar, grav_at_1bar, metalli, grid_resolution,T_max):
    #Creates a Pressure-Temperature profile from temperatures at 1 bar. 
    #Temp_at_1bar : The temperature at 1 bar level
    #grav_at_1bar : The gravity at the 1 bar level, please in cgs
    #metalli: metallicit in [M/H]
    #grid_resolution : how much space you want 
    #T_max : at some temperatures our later simulations will fail, therefor define a T_max

    Press_1bar_to_cgs = 10**6 #to dyne/cm^2, this is Pressure of 1bar in dyne/cm^2
    
    #Step 1: deduce optical depth at 76K level, the Uranus we know of today 
    optical_depth_76K = optical_depth_finder(Press_1bar_to_cgs, 76, metalli,grav_at_1bar)
#    print(f"Optical depth at 76K in Uranus as of today: {optical_depth_76K}")
    
    #Step 2: deduce an estimation for gamma 
    gammas = np.logspace(-2,0,1000)
    T_eff_final = 59.1 
    T_eq_final = 58.2
    T_int_final = (T_eff_final**(4) - T_eq_final**(4))**(1/4)
    final_gamma = find_gamma(optical_depth_76K,T_int_final, gammas, 76, 0.3)
#    print(f"The gamma to be used is : {final_gamma}")
    
    #Step 3: deduce the optical depth at which our 1bar-level is
    optical_depth_our_case = Press_1bar_to_cgs*opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)/grav_at_1bar
    thermal_opacity = opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)
#    print(f"The optical depth at 1 bar in our model: {optical_depth_our_case}")

    #Step 4: deduce which intrinsic Temperature is needed to recover this optical depth at this Pressure and Temperature
    T_int_lin = np.linspace(29,500,1000)
    Tint = find_Tint(optical_depth_our_case,T_int_lin, final_gamma, Temp_at_1bar, 50)
#    print(f"The intrinsic temperature {Tint} in our model")

    #Step 5: Using an upper and lower limit on temperature, find the max optical depth and minimal optical depth
    possible_depths = np.logspace(-10,10,1000)
    final_optical_depth  = find_tau(possible_depths,Tint, final_gamma, T_max, 1e5)
#    print(f"The optical depth until we go {final_optical_depth}")

    #Step 6, over the different optical depths, find all the temperatures
    all_optical_depths = np.logspace(-1,np.log10(final_optical_depth),grid_resolution)
    All_Temperatures = Temp_profile(all_optical_depths, Tint,final_gamma)
#    print(All_Temperatures)
    All_pressures = []
    #Step 7, at different optical depths find different pressures
    for i in range(len(all_optical_depths)):
        aze = pressure_finder_given_opacity(all_optical_depths[i], thermal_opacity, grav_at_1bar)/(10**6)
        #aze = pressure_finder(all_optical_depths[i], All_Temperatures[i], metalli, grav_at_1bar, 1e13)/(10**6)
        All_pressures.append(aze)
#        print(f"At optical depth {all_optical_depths[i]} and Temperature {All_Temperatures[i]}K we have pressures {aze}")

    #Extra Step ~ Find effective temperature 
    T_eq = 58.2 #pm 1 The albedo, effective temperature, and energy balance of Uranus, as determined from Voyager IRIS data
    T_eff_end = (Tint**(4) + T_eq**(4))**(1/4)
    return All_Temperatures, All_pressures, Tint, T_eff_end

In [11]:
def Pressure_at_Temperature_From_1_bar(Temp_at_1bar, grav_at_1bar, metalli, T_forpress):
    #Creates a Pressure-Temperature profile from temperatures at 1 bar and then tries to find the Pressure at a given temperature
    #Temp_at_1bar : The temperature at 1 bar level
    #grav_at_1bar : The gravity at the 1 bar level, please in cgs
    #metalli: metallicit in [M/H]
    #grid_resolution : how much space you want 
    #T_max : at some temperatures our later simulations will fail, therefor define a T_max

    Press_1bar_to_cgs = 10**6 #to dyne/cm^2, this is Pressure of 1bar in dyne/cm^2
    
    #Step 1: deduce optical depth at 76K level, the Uranus we know of today 
    optical_depth_76K = optical_depth_finder(Press_1bar_to_cgs, 76, metalli,grav_at_1bar)
#    print(f"Optical depth at 76K in Uranus as of today: {optical_depth_76K}")
    
    #Step 2: deduce an estimation for gamma 
    gammas = np.logspace(-2,0,1000)
    T_eff_final = 59.1 
    T_eq_final = 58.2
    T_int_final = (T_eff_final**(4) - T_eq_final**(4))**(1/4)
    final_gamma = find_gamma(optical_depth_76K,T_int_final, gammas, 76, 0.3)
#    print(f"The gamma to be used is : {final_gamma}")
    
    #Step 3: deduce the optical depth at which our 1bar-level is
    optical_depth_our_case = Press_1bar_to_cgs*opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)/grav_at_1bar
    thermal_opacity = opacity(Press_1bar_to_cgs, Temp_at_1bar, metalli)
#    print(f"The optical depth at 1 bar in our model: {optical_depth_our_case}")

    #Step 4: deduce which intrinsic Temperature is needed to recover this optical depth at this Pressure and Temperature
    T_int_lin = np.linspace(29,500,1000)
    Tint = find_Tint(optical_depth_our_case,T_int_lin, final_gamma, Temp_at_1bar, 50)
#    print(f"The intrinsic temperature {Tint} in our model")

    #Step 5: Using an upper and lower limit on temperature, find the max optical depth and minimal optical depth
    possible_depths = np.logspace(-10,10,1000)
    final_optical_depth  = find_tau(possible_depths,Tint, final_gamma, T_forpress, 1e5)
#    print(f"The optical depth until we go {final_optical_depth}")

    aze = pressure_finder_given_opacity(final_optical_depth, thermal_opacity, grav_at_1bar)/(10**6)
    return aze

def Find_atm_T(P, T, met, surf_grav):
    def test_find_atm_T(atmo_T):
        return Pressure_at_Temperature_From_1_bar(atmo_T, surf_grav, met,T) - P
    T_atmo_1bar = fsolve(test_find_atm_T, 1e2)
    return T_atmo_1bar[0]
def Pressure_and_Temperature_to_profile(P, T, met, surf_grav,grid_resolution):
    #Creates pressure and temperature profile given one pressure and temperature
    #optical depths 
    #T in Kelvin
    #met in [M/H], different then how we define elsewhere
    # the surface gravity
    #P in bar
    T_atmo_1bar = Find_atm_T(P, T, met, surf_grav)
    All_Temperatures, All_pressures, Tint, T_eff_end = Profile_From_1_bar(T_atmo_1bar, surf_grav, met, grid_resolution,T)
    return All_Temperatures, All_pressures, Tint, T_eff_end, T_atmo_1bar

In [12]:
def Find_T_from_atmo_and_Pressure(T_at_1bar, Pressure_at_layer, met, surf_grav):
    #Finds the Temperature when given atmospheric temperature at 1 bar and pressure
    #T_at_1bar, the temperature at the 1 bar level
    #The pressure at our starting layer,
    #met metallicity as in [M/H] 
    #surf_grav, surface gravity woop

    Press_1bar_to_cgs = 10**6 #to dyne/cm^2, this is Pressure of 1bar in dyne/cm^2
    
    #Step 1: deduce optical depth at 76K level, the Uranus we know of today 
    optical_depth_76K = optical_depth_finder(Press_1bar_to_cgs, 76, met,surf_grav)
#    print(f"Optical depth at 76K in Uranus as of today: {optical_depth_76K}")
    
    #Step 2: deduce an estimation for gamma 
    gammas = np.logspace(-2,0,1000)
    T_eff_final = 59.1 
    T_eq_final = 58.2
    T_int_final = (T_eff_final**(4) - T_eq_final**(4))**(1/4)
    final_gamma = find_gamma(optical_depth_76K,T_int_final, gammas, 76, 0.3)
#    print(f"The gamma to be used is : {final_gamma}")
    
    #Step 3: deduce the optical depth at which our 1bar-level is
    optical_depth_our_case = Press_1bar_to_cgs*opacity(Press_1bar_to_cgs, T_at_1bar, met)/surf_grav
    thermal_opacity = opacity(Press_1bar_to_cgs, T_at_1bar, met)
#    print(f"The optical depth at 1 bar in our model: {optical_depth_our_case}")

    #Step 4: deduce which intrinsic Temperature is needed to recover this optical depth at this Pressure and Temperature
    T_int_lin = np.linspace(29,500,1000)
    Tint = find_Tint(optical_depth_our_case,T_int_lin, final_gamma, T_at_1bar, 50)
#    print(f"The intrinsic temperature {Tint} in our model")

    #Step 5: Deduce the optical depth needed for our pressure
    pressure_tau = optical_depth_finder_given_opacity(Pressure_at_layer*10**6, thermal_opacity, surf_grav)
    #Step 6: Given everything now, deduce the temperature at our layer
    Final_T = Temp_profile(pressure_tau, Tint, final_gamma)
    return Final_T

def Create_Temperature_Constant_Pressure(Constant_Pressure, starting_Temperature_at_layer, ending_Temperature_1bar, met, surf_grav, timesteps,grid_resolution):
    #Here we start from a temperature with a pressure there. Next we construct how that pressure evolves in time, how it cools down, while also solving
    # temperature-pressure profiles at all different timesteps
    # Tatmo returns a list of 1 bar temperatures during the timesteps
    # Solidus returns wether at that timestep we still have melt or not 
    # Tlayer returns temperatures at those layers
    # Pressures_atmo, Temperatures_atmo return the pressure-temperature profile

    #Constant Pressure: The pressure found back at our layer in bar
    #starting_Temperature_at_layer: the temperature at our layer 
    #ending_Temperature_1bar: the ending temperature, taken often 76 K
    # met metallicity as in [M/H], often taken 1.5
    #surf_grav in cgs, often 887
    #timesteps, the amount of steps in time
    #gridresolution : how many values in our atmosphere

    #Step 1: from our pressure and layer, construct a temperature-pressure profile, from this we can also get pressure at 1 bar
    Temperature_profile0, Pressure_profile0, Tint0, T_eff0, T_atmo_1bar0 = Pressure_and_Temperature_to_profile(Constant_Pressure, starting_Temperature_at_layer, met, surf_grav,grid_resolution)
    
    #Step 2: create a linspace of temperatures from then to now
    T_atmo_lin = np.linspace(T_atmo_1bar0,ending_Temperature_1bar, timesteps)

    #Step 3: Find the layers temperatures for the different 1 bar temperatures
    Tlayer = []
    for Ts in T_atmo_lin:
        Tlayer.append(Find_T_from_atmo_and_Pressure(Ts, Constant_Pressure, met, surf_grav))
    Tlayer = np.array(Tlayer)

    #Step 4: Consctruct from the different layers different pressure temperature profiles, the final state, the state as of today often gives
    #numerical errors, so we exclude it first and then calculate
    T_profiles = np.zeros((timesteps, grid_resolution))
    P_profiles = np.zeros((timesteps, grid_resolution))
    Tint_profiles = np.zeros((timesteps, 1))
    T_eff_profiles = np.zeros((timesteps, 1))
    T_atmo_1bar_profiles = np.zeros((timesteps, 1))
    Melt_profiles = np.zeros((timesteps, 1))
    
    for i in range(len(Tlayer)-1):
        Ti, Pi, Tinti, T_effi, T_atmo_1bari = Pressure_and_Temperature_to_profile(Constant_Pressure, Tlayer[i], met, surf_grav,grid_resolution)
        T_profiles[i] = Ti
        P_profiles[i] = Pi
        Tint_profiles[i] = Tinti
        T_eff_profiles[i] = T_effi
        T_atmo_1bar_profiles[i] = T_atmo_1bari

    #Step 5 construct the final layer
    T_profiles[-1], P_profiles[-1], Tint_profiles[-1], T_eff_profiles[-1] = Profile_From_1_bar(ending_Temperature_1bar, surf_grav, met, grid_resolution,Tlayer[-1])
    T_atmo_1bar_profiles[-1] = ending_Temperature_1bar

    #Step 6, for our starting layers, conclude wether magma ocean or not with solidus
    Solidus_line = peridotite_solidus(Constant_Pressure)
    for i in range(len(Tlayer)):
        if Tlayer[i] < Solidus_line:
            Melt_profiles[i] = False
        else:
            Melt_profiles[i] = True
    return Tlayer, T_atmo_1bar_profiles,T_profiles, P_profiles, Tint_profiles, T_eff_profiles, Melt_profiles

In [13]:
def give_melt(Pressure, Temperature):
    #Give pressure in bar
    Tline = peridotite_solidus(Pressure)
    if Tline <= Temperature:
        return 1
    elif Tline> Temperature:
        return 0

def create_melt_list(excel_file):
    #Excel file from atmodeller to generate a list with values for 0 or 1
    mantle_melt = []
    Pressures = pd.read_excel(excel_file, sheet_name="state")["pressure"]
    Temperatures = pd.read_excel(excel_file, sheet_name="state")["temperature"]

    for i in range(len(Pressures)):
        mantle_melt.append(give_melt(Pressures[i], Temperatures[i]))


    return mantle_melt

In [14]:
def create_phase_lookup(filename):
    """
    Create a simple phase lookup function
    """
    # Load data
    data = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                vals = line.split()
                float(vals[0])  # Test if numeric
                data.append([float(x) for x in vals[:11]])
            except (ValueError, IndexError):
                continue
    
    data = np.array(data)
    
    # Phase names
    phase_names = {
        -1: "Ice Ih", -2: "Ice II", -3: "Ice III",
        -5: "Ice V", -6: "Ice VI", -7: "Ice VII",
        3: "Vapor", 4: "Liquid", 5: "Supercritical"
    }
    
    # Extract columns
    temps = data[:, 1]      # Temperature in K
    press = data[:, 0] / 1e9  # Pressure in GPa
    phases = data[:, 10]     # Phase ID
    
    def lookup(T, P):
        """Return phase name at T(K) and P(GPa)"""
        # Find closest point
        distances = (temps - T)**2 + (press - P)**2
        idx = np.argmin(distances)
        phase_id = int(phases[idx])
        return phase_names.get(phase_id, f"Phase {phase_id}")
    
    return lookup

filename = '/home/mathias/H2Oeq/AQUA/Tables/aqua_eos_pt_v1_0.dat'
get_phase = create_phase_lookup(filename)

In [15]:
def read_aqua_simple(filename):
    """
    Read AQUA data file, skipping header lines that start with text
    """
    data = []
    line_count = 0
    skipped_count = 0
    
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            line_count += 1
            
            # Skip empty lines
            if not line:
                continue
            
            # Try to convert to float - if it fails, skip the line
            try:
                # Split and try to convert first element to float
                vals = line.split()
                if vals:  # Check if line has content
                    float(vals[0])  # Test if first value is a number
                    # If successful, convert all values to float
                    data.append([float(x) for x in vals[:11]])
            except ValueError:
                # This line contains text - skip it
                skipped_count += 1
                continue
    
    print(f"Read {line_count} total lines")
    print(f"Skipped {skipped_count} header/comment lines")
    print(f"Loaded {len(data)} data points")
    
    if len(data) == 0:
        raise ValueError("No data points were loaded! Check file format.")
    
    # Convert to DataFrame
    df = pd.DataFrame(data, columns=[
        'press_Pa', 'temp_K', 'rho', 'ad_grad', 's', 'u', 
        'c', 'mmw', 'x_ion', 'x_d', 'phase_id'
    ])
    
    # Convert pressure to GPa for convenience
    df['press_GPa'] = df['press_Pa'] / 1e9
    
    # Ensure phase_id is integer
    df['phase_id'] = df['phase_id'].astype(int)
    
    # Phase names mapping
    phase_names = {
        -1: "Ice Ih",
        -2: "Ice II",
        -3: "Ice III",
        -5: "Ice V",
        -6: "Ice VI",
        -7: "Ice VII",
        -10: "Ice X",
        3: "Vapor",
        4: "Liquid",
        5: "Supercritical/Superionic"
    }
    
    def get_phase(T, P):
        """Get phase at temperature T(K) and pressure P(GPa)"""
        # Find closest point
        distances = np.sqrt(
            (df['temp_K'] - T)**2 + 
            (df['press_GPa'] - P)**2
        )
        idx = distances.idxmin()
        pid = df.loc[idx, 'phase_id']
        
        # Also return density and actual T,P for verification
        actual_T = df.loc[idx, 'temp_K']
        actual_P = df.loc[idx, 'press_GPa']
        density = df.loc[idx, 'rho']
        
        phase_name = phase_names.get(pid, f"Unknown Phase ({pid})")
        
        return {
            'phase': phase_name,
            'phase_id': pid,
            'density': density,
            'actual_T': actual_T,
            'actual_P': actual_P,
            'distance': distances[idx]
        }
    
    # Print summary
    print(f"\n✅ Successfully loaded {len(df):,} data points")
    print(f"   Temperature range: {df['temp_K'].min():.0f} - {df['temp_K'].max():.0f} K")
    print(f"   Pressure range: {df['press_GPa'].min():.6f} - {df['press_GPa'].max():.2f} GPa")
    print(f"   Phase IDs present: {sorted(df['phase_id'].unique())}")
    
    return df, get_phase

# Alternative: Using pandas with comment handling
def read_aqua_pandas(filename):
    """
    Alternative method using pandas with comment handling
    """
    # Try to find where the numeric data starts
    with open(filename, 'r') as f:
        for i, line in enumerate(f):
            try:
                # Try to parse first few numbers
                vals = line.strip().split()
                if vals:
                    float(vals[0])
                    float(vals[1])
                    # If successful, this is the start of data
                    skip_rows = i
                    print(f"Data starts at line {i+1}")
                    break
            except (ValueError, IndexError):
                continue
    
    # Read with pandas, skipping header rows
    df = pd.read_csv(
        filename,
        sep=r'\s+',
        skiprows=skip_rows,
        names=['press_Pa', 'temp_K', 'rho', 'ad_grad', 's', 'u', 
               'c', 'mmw', 'x_ion', 'x_d', 'phase_id'],
        engine='python'
    )
    
    df['press_GPa'] = df['press_Pa'] / 1e9
    df['phase_id'] = df['phase_id'].astype(int)
    
    return df

# ===== USAGE =====

filename = '/home/mathias/H2Oeq/AQUA/Tables/aqua_eos_pt_v1_0.dat'

# Method 1: Simple function
print("="*60)
print("LOADING AQUA DATA")
print("="*60)
df, get_phase = read_aqua_simple(filename)

# Test points
print("\n" + "="*60)
print("TEST POINTS")
print("="*60)

test_points = [
    (300, 0.0001),   # Near surface
    (500, 1),         # 1 GPa
    (1000, 10),       # 10 GPa
    (2000, 50),       # 50 GPa
    (4000, 100),      # 100 GPa
]

for T, P in test_points:
    result = get_phase(T, P)
    print(f"T={T:4.0f}K, P={P:6.2f}GPa → {result['phase']:20s} "
          f"(ρ={result['density']:8.1f} kg/m³) "
          f"[closest point: T={result['actual_T']:.0f}K, P={result['actual_P']:.3f}GPa]")

# Quick lookup function for single use
def quick_phase(T, P):
    """Quick one-line phase lookup"""
    return get_phase(T, P)['phase']


LOADING AQUA DATA
Read 329012 total lines
Skipped 19 header/comment lines
Loaded 328993 data points

✅ Successfully loaded 328,993 data points
   Temperature range: 100 - 100000 K
   Pressure range: 0.000000 - 400000.00 GPa
   Phase IDs present: [np.int64(-10), np.int64(-7), np.int64(-6), np.int64(-5), np.int64(-3), np.int64(-2), np.int64(-1), np.int64(3), np.int64(4), np.int64(5)]

TEST POINTS
T= 300K, P=  0.00GPa → Liquid               (ρ=   996.0 kg/m³) [closest point: T=302K, P=0.000GPa]
T= 500K, P=  1.00GPa → Liquid               (ρ=  1139.4 kg/m³) [closest point: T=501K, P=1.003GPa]
T=1000K, P= 10.00GPa → Supercritical/Superionic (ρ=  1618.3 kg/m³) [closest point: T=1000K, P=10.034GPa]
T=2000K, P= 50.00GPa → Supercritical/Superionic (ρ=  2338.4 kg/m³) [closest point: T=1995K, P=50.297GPa]
T=4000K, P=100.00GPa → Supercritical/Superionic (ρ=  2724.1 kg/m³) [closest point: T=3981K, P=100.366GPa]


In [16]:
def list_phases(Temperatures, Pressures):
    #pressures in Gpa, Temperatures in kelvin
    phases = []
    for i in range(len(Temperatures)):
        phase = quick_phase(Temperatures[i], Pressures[i])
        phases.append(phase)
        phases = list(set(phases))
    return phases

def list_phases_index(Temperatures, Pressures):
    #pressures in Gpa, Temperatures in kelvin
    phases = []
    for i in range(len(Temperatures)):
        phase = quick_phase(Temperatures[i], Pressures[i])
        phases.append(phase)
    return phases

def phase_lists(Temperatures, Pressures): 
    P_condensed = []
    T_condensed = []
    P_gas = []
    T_gas = []

    for i in range(len(Temperatures)):
        phase = quick_phase(Temperatures[i], Pressures[i])
        if phase=='Ice Ih' or phase=='Liquid':
            P_condensed.append(Pressures[i])
            T_condensed.append(Temperatures[i])

        elif phase=='Vapor' or phase=='Supercritical/Superionic':
            P_gas.append(Pressures[i])
            T_gas.append(Temperatures[i])

    P_condensed = np.array(P_condensed)
    T_condensed = np.array(T_condensed)
    P_gas = np.array(P_gas)
    T_gas = np.array(T_gas)
    
    idx_condensed = np.argsort(-T_condensed)
    idx_gas = np.argsort(-T_gas)

    P_condensed = P_condensed[idx_condensed]
    T_condensed = T_condensed[idx_condensed]
    P_gas = P_gas[idx_gas]
    T_gas = T_gas[idx_gas]

    return P_condensed, T_condensed, P_gas, T_gas
    

In [22]:
filename = f"/home/mathias/Margtest/Magrathea/result_final0/results.xlsx"
params = pd.read_excel(filename, sheet_name="Run_Parameters")
mask = params["mass_of_hydro"] == 0
Radii = pd.read_excel(filename, sheet_name="Results")["Radius (earth)"][mask]
Masses = pd.read_excel(filename, sheet_name="Results")["M (earth)"][mask]
Temperatures_layer = pd.read_excel(filename, sheet_name="Results")["T (K)"][mask]
Pressures_layer = pd.read_excel(filename, sheet_name="Results")["P (GPa)"][mask]


Radii = Radii.iloc[::-1].reset_index(drop=True)
Masses = Masses.iloc[::-1].reset_index(drop=True)
Temperatures_layer = Temperatures_layer.iloc[::-1].reset_index(drop=True)


Radii_atm = Radii[::25]
Temperatures_layer_atm = Temperatures_layer[::25]
Masses_atm = Masses[::25]
print(Temperatures_layer_atm)

0      2617.27940
25     2464.96050
50     2325.90580
75     2185.06900
100    2036.07940
125    1880.29960
150    1699.31020
175    1460.72430
200    1052.59330
225     887.99524
250     607.76130
Name: T (K), dtype: float64


In [23]:
hmps_atm = hmps[::5]
print(hmps_atm)

[0.1               0.12648552168553  0.159985871960606 0.202358964772516
 0.255954792269954 0.323745754281764 0.409491506238042 0.517947467923121
 0.655128556859551 0.828642772854684]


In [20]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1

#model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
#model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)
model_atm_sol_real = EquilibriumModel(species_HOSi_atm_sol_real)
rho_centre = 3000
fO2_log10_shifts = [-3,-2,-1,0]
for k in range(len(fO2_log10_shifts)):
    for i in range(len(Radii_atm)):
        if Temperatures_layer_atm.iloc[i] > 700:
            radius = Radii_atm.iloc[i]*R_earth
            masses = Masses_atm.iloc[i]*M_earth
            temp_layer = Temperatures_layer_atm.iloc[i]
            shift = f"{fO2_log10_shifts[k]:.2g}"

            for j in range(len(hmps_atm)):
                hmp = hmps_atm[-1]
                hmp = 1
                hmp_text = f"{hmp:.3g}"
                file_name = f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
                P = pd.read_excel(file_name, sheet_name="state")["pressure"]
                T = pd.read_excel(file_name, sheet_name="state")["temperature"]
                All_Temperatures, All_Pressures, Tint, T_eff_end, T_atmo_1bar = Pressure_and_Temperature_to_profile(P[-1], T[-1], 1.5, 887,100)
                CondensedP, CondensedT, GasP, GasT = phase_lists(All_Temperatures, np.array(All_Pressures)*10**(-4))

                CondensedP_bar = np.array(CondensedP)*10**4
                GasP_bar = np.array(GasP)*10**4

                mask = np.array(CondensedT) > 250
                CondensedT = np.array(CondensedT)[mask]
                CondensedP_bar = CondensedP_bar[mask]
                
                atm_scalings_condensed = CondensedP_bar/max(All_Pressures)
                atm_scalings_gas = GasP_bar/max(All_Pressures)
                # Magma - Solubility - Real Gas
        
                H_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_H")[
                    "gas_mass"
                ][-1]
                Si_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_Si")[
                    "gas_mass"
                ][-1]
                O_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_O")[
                    "gas_mass"
                ][-1]

                mass_constraints_condensed = {
                    "H": H_mass_atm_magma_sol_real * atm_scalings_condensed,
                    "O": O_mass_atm_magma_sol_real * atm_scalings_condensed,
                    "Si": Si_mass_atm_magma_sol_real * atm_scalings_condensed,
                }

                mass_constraints_gas = {
                    "H": H_mass_atm_magma_sol_real * atm_scalings_gas,
                    "O": O_mass_atm_magma_sol_real * atm_scalings_gas,
                    "Si": Si_mass_atm_magma_sol_real * atm_scalings_gas,
                }

                state_gas: ThermodynamicStateProtocol = ThermodynamicState(GasT, GasP_bar)
                #Calculating the gaseous part for H2O
                model_atm_sol_real.solve(
                    state=state_gas,
                    mass_constraints=mass_constraints_gas,
                    solver_parameters=SolverParameters(multistart=20),
                )
                output_atm_magma_sol_real_gas = model_atm_sol_real.output

                output_atm_magma_sol_real_gas.quick_look()
    
                output_atm_magma_sol_real_gas.to_excel(
                    f"./TEST_real_gas"
                )
                output_atm_magma_sol_real_gas.to_pickle(
                    f"./TEST_real_gas"
                )

                state_condensed: ThermodynamicStateProtocol = ThermodynamicState(CondensedT, CondensedP_bar)
                #Calculating the condensed part for H2O
                model_atm_sol_real.solve(
                    state=state_condensed,
                    mass_constraints=mass_constraints_condensed,
                    solver_parameters=SolverParameters(multistart=20),
                )
                output_atm_magma_sol_real_condensed = model_atm_sol_real.output

                output_atm_magma_sol_real_condensed.quick_look()
    
                output_atm_magma_sol_real_condensed.to_excel(
                    f"./TEST_real_condensed"
                )
                output_atm_magma_sol_real_condensed.to_pickle(
                    f"./TEST_real_condensed"
                )
                

[13:04:51 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, SolubilityPowerLaw', 'H2_g: CombinedRealGas, SolubilityPowerLawLog10', 'O2_g: CombinedRealGas, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[13:04:51 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[13:04:51 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}


NameError: name 'species_HOSi_atm_sol_real' is not defined

In [ ]:
type(output_atm_magma_sol_real_gas)

In [25]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1

model_atm_sol_real_gas = EquilibriumModel(species_HOSi_atm_sol_real_gas)
model_atm_sol_real_condensed = EquilibriumModel(species_HOSi_atm_sol_real_condensed)
rho_centre = 3000
fO2_log10_shifts = [-3]
for k in range(len(fO2_log10_shifts)):
    for i in range(len(Radii_atm[0:2])):
        if Temperatures_layer_atm.iloc[i] > 700:
            radius = Radii_atm.iloc[i]*R_earth
            masses = Masses_atm.iloc[i]*M_earth
            temp_layer = Temperatures_layer_atm.iloc[i]
            shift = f"{fO2_log10_shifts[k]:.2g}"

            for j in range(len(hmps_atm[0:2])):
                hmp = hmps_atm[j]
                hmp_text = f"{hmp:.3g}"
                file_name = f"./excel_Uranus_time/HOSi/rho{rho_centre}/core_boundary/HOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
                P = pd.read_excel(file_name, sheet_name="state")["pressure"]
                T = pd.read_excel(file_name, sheet_name="state")["temperature"]
                All_Temperatures, All_Pressures, Tint, T_eff_end, T_atmo_1bar = Pressure_and_Temperature_to_profile(P.iloc[-1], T.iloc[-1], 1.5, 887,100)
                CondensedP, CondensedT, GasP, GasT = phase_lists(All_Temperatures, np.array(All_Pressures)*10**(-4))

                CondensedP_bar = np.array(CondensedP)*10**4
                GasP_bar = np.array(GasP)*10**4

                mask = np.array(CondensedT) > 250
                CondensedT = np.array(CondensedT)[mask]
                CondensedP_bar = CondensedP_bar[mask]
                
                atm_scalings_condensed = CondensedP_bar/max(All_Pressures)
                atm_scalings_gas = GasP_bar/max(All_Pressures)
                # Magma - Solubility - Real Gas
        
                H_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_H")[
                    "gas_mass"
                ].iloc[-1]
                Si_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_Si")[
                    "gas_mass"
                ].iloc[-1]
                O_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_O")[
                    "gas_mass"
                ].iloc[-1]

                total_gas_mass = H_mass_atm_magma_sol_real + Si_mass_atm_magma_sol_real + O_mass_atm_magma_sol_real
                H_fraction = H_mass_atm_magma_sol_real/total_gas_mass 
                O_fraction = O_mass_atm_magma_sol_real/total_gas_mass 
                Si_fraction = Si_mass_atm_magma_sol_real/total_gas_mass 
                
                total_gas_mass_scaled = atm_scalings_gas*total_gas_mass
                total_condensed_mass_scaled = atm_scalings_condensed*total_gas_mass
                

#First the gaseous H2O part, then later H2O condensed -> rainout for exoplanets? search literature
                outputs = {}
                for i in range(len(GasT)):
                    print(GasT[i])
                    state_gas: ThermodynamicStateProtocol = ThermodynamicState(GasT[i], GasP_bar[i])
                    mass_constraints_gas = {
                    "H": H_fraction * total_gas_mass_scaled[i],
                    "O": O_fraction * total_gas_mass_scaled[i],
                    "Si": Si_fraction * total_gas_mass_scaled[i],
                    }
                    #Calculating the gaseous part for H2O
                    model_atm_sol_real_gas.solve(
                        state=state_gas,
                        mass_constraints=mass_constraints_gas,
                        solver_parameters=SolverParameters(multistart=20),
                    )
                    output_atm_sol_real_gas = model_atm_sol_real_gas.output
                    dfs = model_atm_sol_real_gas.output.to_dataframes()
    
                    for sheet_name, df in dfs.items():
                        df = df.copy() 
                        df["run_id"] = i
        
                        if sheet_name not in outputs:
                            outputs[sheet_name] = []
        
                        outputs[sheet_name].append(df)
                    
                    gas_dict = output_atm_sol_real_gas.element_moles_gas()
                    Hydrogen_number = (gas_dict[0])[0]
                    H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                    Oxygen_number = (gas_dict[0])[1]
                    O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                    Silicon_number = (gas_dict[0])[2]
                    Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                    total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real
                    H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                    O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                    Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                    output_atm_sol_real_gas.quick_look()
                    
                               

                combined = {
                    sheet: pd.concat(df_list, ignore_index=True)
                    for sheet, df_list in outputs.items()
                    }

                # Write to Excel with SAME sheet names
                with pd.ExcelWriter("TEST_real_gas.xlsx") as writer:
                    for sheet_name, df in combined.items():
                        df.to_excel(writer, sheet_name=sheet_name, index=False)

#Now gaseous part 
                outputs = {}
                for i in range(len(CondensedT)):
                    if CondensedT[i] > 0:
                        print(CondensedT[i])
                        state_gas: ThermodynamicStateProtocol = ThermodynamicState(CondensedT[i], CondensedP_bar[i])
                        mass_constraints_gas = {
                        "H": H_fraction * total_condensed_mass_scaled[i],
                        "O": O_fraction * total_condensed_mass_scaled[i],
                        "Si": Si_fraction * total_condensed_mass_scaled[i],
                        }
                        #Calculating the gaseous part for H2O
                        model_atm_sol_real_condensed.solve(
                            state=state_gas,
                            mass_constraints=mass_constraints_gas,
                            solver_parameters=SolverParameters(multistart=20),
                        )
                        output_atm_sol_real_condensed = model_atm_sol_real_condensed.output
                        dfs = model_atm_sol_real_condensed.output.to_dataframes()
    
                        for sheet_name, df in dfs.items():
                            df = df.copy() 
                            df["run_id"] = i
        
                            if sheet_name not in outputs:
                                outputs[sheet_name] = []
        
                            outputs[sheet_name].append(df)
                    
                        gas_dict = output_atm_sol_real_condensed.element_moles_gas()
                        Hydrogen_number = (gas_dict[0])[0]
                        H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                        Oxygen_number = (gas_dict[0])[1]
                        O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                        Silicon_number = (gas_dict[0])[2]
                        Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                        total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real
                        H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                        O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                        Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                        output_atm_sol_real_condensed.quick_look()
    
                        output_atm_sol_real_condensed.to_excel(
                            f"./TEST_real_condensed"
                        )
                        output_atm_sol_real_condensed.to_pickle(
                            f"./TEST_real_condensed"
                        )

                    combined = {
                        sheet: pd.concat(df_list, ignore_index=True)
                        for sheet, df_list in outputs.items()
                        }

                # Write to Excel with SAME sheet names
                with pd.ExcelWriter("TEST_real_condensed.xlsx") as writer:
                    for sheet_name, df in combined.items():
                        df.to_excel(writer, sheet_name=sheet_name, index=False)

[13:10:10 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, NoSolubility', 'H2_g: CombinedRealGas, NoSolubility', 'O2_g: CombinedRealGas, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[13:10:10 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[13:10:10 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[13:10:10 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, NoSolubility', 'H2_g: CombinedRealGas, NoSolubility', 'O2_g: CombinedRealGas, NoSolubility', 'H2O_cd: CondensateActivity, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'O2Si_cd: Condensat

2617.279399999995


[13:10:51 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:10:51 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:10:51 - atmodeller.classes             - INFO     ] - Solver steps (max) = 70
[13:10:51 - atmodeller.output_core         - DEBUG    ] - Creating Output
[13:10:51 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[13:10:51 - atmodeller.output_core         - INFO     ] - Computing asdict output
[13:10:51 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[13:10:52 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:10:52 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:10:52 - atmodeller.classes             - INFO     ] - Solver steps (max) = 50
[13:10:52 

2511.271297550032


[13:10:53 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[13:10:54 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:10:54 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:10:54 - atmodeller.classes             - INFO     ] - Solver steps (max) = 50
[13:10:54 - atmodeller.output_core         - DEBUG    ] - Creating Output
[13:10:54 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[13:10:54 - atmodeller.output_core         - INFO     ] - Computing asdict output


2409.556879102595


... (153 outputs left out)

1053.9301390962473


[13:14:31 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[13:14:32 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:14:32 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:14:32 - atmodeller.classes             - INFO     ] - Solver steps (max) = 102
[13:14:32 - atmodeller.output_core         - DEBUG    ] - Creating Output
[13:14:32 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[13:14:32 - atmodeller.output_core         - INFO     ] - Computing asdict output


1011.244378327987


KeyboardInterrupt: 

### All condensed species

In [29]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1
hmp = 1
model_atm_sol_real_gas = EquilibriumModel(species_HHeCNOSi_atmosphere_gas)
model_atm_sol_real_condensed = EquilibriumModel(species_HHeCNOSi_atmosphere_condensed)
model_atm_sol_real_gas_noSiO2 = EquilibriumModel(species_HHeCNOSi_atmosphere_gas_noSiO2)
rho_centre = 3000
fO2_log10_shifts = [-3]
for k in range(len(fO2_log10_shifts)):
    for i in range(len(Radii_atm)):
        if Temperatures_layer_atm.iloc[i] > 700:
            radius = Radii_atm.iloc[i]*R_earth
            masses = Masses_atm.iloc[i]*M_earth
            temp_layer = Temperatures_layer_atm.iloc[i]
            shift = f"{fO2_log10_shifts[k]:.2g}"
            hmp_text = f"{hmp:.3g}"
            file_name = f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
            P = pd.read_excel(file_name, sheet_name="state")["pressure"]
            T = pd.read_excel(file_name, sheet_name="state")["temperature"]
            All_Temperatures, All_Pressures, Tint, T_eff_end, T_atmo_1bar = Pressure_and_Temperature_to_profile(P.iloc[-1], T.iloc[-1], 1.5, 887,100)
            CondensedP, CondensedT, GasP, GasT = phase_lists(All_Temperatures, np.array(All_Pressures)*10**(-4))

            CondensedP_bar = np.array(CondensedP)*10**4
            GasP_bar = np.array(GasP)*10**4

            mask = np.array(CondensedT) > 250
            CondensedT = np.array(CondensedT)[mask]
            CondensedP_bar = CondensedP_bar[mask]
                
            atm_scalings_condensed = CondensedP_bar/max(All_Pressures)
            atm_scalings_gas = GasP_bar/max(All_Pressures)
                # Magma - Solubility - Real Gas
        
            H_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_H")[
                "gas_mass"
            ].iloc[-1]
            Si_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_Si")[
                "gas_mass"
            ].iloc[-1]
            O_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_O")[
                "gas_mass"
            ].iloc[-1]
            He_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_He")[
                "gas_mass"
            ].iloc[-1]
            C_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_C")[
                "gas_mass"
            ].iloc[-1]
            N_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_N")[
                "gas_mass"
            ].iloc[-1]


            total_gas_mass = H_mass_atm_magma_sol_real + Si_mass_atm_magma_sol_real + O_mass_atm_magma_sol_real + He_mass_atm_magma_sol_real + N_mass_atm_magma_sol_real + C_mass_atm_magma_sol_real
                
                 
            H_fraction = H_mass_atm_magma_sol_real/total_gas_mass 
            O_fraction = O_mass_atm_magma_sol_real/total_gas_mass 
            Si_fraction = Si_mass_atm_magma_sol_real/total_gas_mass
            He_fraction = He_mass_atm_magma_sol_real/total_gas_mass 
            C_fraction = C_mass_atm_magma_sol_real/total_gas_mass 
            N_fraction = N_mass_atm_magma_sol_real/total_gas_mass 
                
            total_gas_mass_scaled = atm_scalings_gas*total_gas_mass
            total_condensed_mass_scaled = atm_scalings_condensed*total_gas_mass

            state_gas: ThermodynamicStateProtocol = ThermodynamicState(GasT[0], GasP_bar[0])
            mass_constraints_gas = {
            "H": H_fraction * total_gas_mass_scaled[0],
            "O": O_fraction * total_gas_mass_scaled[0],
            "Si": Si_fraction * total_gas_mass_scaled[0],
            "He": He_fraction * total_gas_mass_scaled[0],
            "C": C_fraction * total_gas_mass_scaled[0],
            "N": N_fraction * total_gas_mass_scaled[0],
            }
            #Calculating the gaseous part for H2O
            model_atm_sol_real_gas_noSiO2.solve(
                state=state_gas,
                mass_constraints=mass_constraints_gas,
                solver_parameters=SolverParameters(multistart=20),
            )
            output_atm_sol_real_gas = model_atm_sol_real_gas_noSiO2.output
                    
            gas_dict = output_atm_sol_real_gas.element_moles_gas()

            Carbon_number = (gas_dict[0])[0]
            C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
            Hydrogen_number = (gas_dict[0])[1]
            H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

            Helium_number = (gas_dict[0])[2]
            He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

            Nitrogen_number = (gas_dict[0])[3]
            N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

            Oxygen_number = (gas_dict[0])[4]
            O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

            Silicon_number = (gas_dict[0])[5]
            Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

            total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + C_mass_atm_sol_real + N_mass_atm_sol_real
                    
            H_fraction = H_mass_atm_sol_real/total_atm_sol_real
            O_fraction = O_mass_atm_sol_real/total_atm_sol_real
            Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
            He_fraction = He_mass_atm_sol_real/total_atm_sol_real
            C_fraction = C_mass_atm_sol_real/total_atm_sol_real
            N_fraction = N_mass_atm_sol_real/total_atm_sol_real
            output_atm_sol_real_gas.quick_look()

            initial_log_number_moles = output_atm_sol_real_gas.log_number_moles
            initial_log_number_moles = np.concatenate(
                [initial_log_number_moles, [[53.0]]],
                axis=1
                )
                
#you can also set initial moles in solverparameters
#First the gaseous H2O part, then later H2O condensed
            outputs = {}
            for i in range(len(GasT)-1):
                print(GasT[i+1])
                state_gas: ThermodynamicStateProtocol = ThermodynamicState(GasT[i+1], GasP_bar[i+1])
                mass_constraints_gas = {
                "H": H_fraction * total_gas_mass_scaled[i+1],
                "O": O_fraction * total_gas_mass_scaled[i+1],
                "Si": Si_fraction * total_gas_mass_scaled[i+1],
                "He": He_fraction * total_gas_mass_scaled[i+1],
                "C": C_fraction * total_gas_mass_scaled[i+1],
                "N": N_fraction * total_gas_mass_scaled[i+1],
                }
                #Calculating the gaseous part for H2O
                model_atm_sol_real_gas.solve(
                    state=state_gas,
                    mass_constraints=mass_constraints_gas,
                    initial_log_number_moles=initial_log_number_moles,
                    solver_parameters=SolverParameters(multistart=200),
                )
                output_atm_sol_real_gas = model_atm_sol_real_gas.output
                dfs = model_atm_sol_real_gas.output.to_dataframes()
    
                for sheet_name, df in dfs.items():
                    df = df.copy() 
                    df["run_id"] = i
        
                    if sheet_name not in outputs:
                        outputs[sheet_name] = []
        
                    outputs[sheet_name].append(df)
                    
                gas_dict = output_atm_sol_real_gas.element_moles_gas()

                Carbon_number = (gas_dict[0])[0]
                C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
                Hydrogen_number = (gas_dict[0])[1]
                H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                Helium_number = (gas_dict[0])[2]
                He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

                Nitrogen_number = (gas_dict[0])[3]
                N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

                Oxygen_number = (gas_dict[0])[4]
                O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                Silicon_number = (gas_dict[0])[5]
                Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + N_mass_atm_sol_real + C_mass_atm_sol_real
                H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                He_fraction = He_mass_atm_sol_real/total_atm_sol_real
                C_fraction = C_mass_atm_sol_real/total_atm_sol_real
                N_fraction = N_mass_atm_sol_real/total_atm_sol_real
                output_atm_sol_real_gas.quick_look()
                initial_log_number_moles = output_atm_sol_real_gas.log_number_moles
                

            combined = {
                sheet: pd.concat(df_list, ignore_index=True)
                for sheet, df_list in outputs.items()
                }


            # Write to Excel with SAME sheet names
            with pd.ExcelWriter(f"./excel_Uranus_time/HHeCNOSi/Precipiation/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}_hmps1_gas_allcondensed.xlsx") as writer:
                for sheet_name, df in combined.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

#Now gaseous part 
            initial_log_number_moles = np.concatenate(
                [initial_log_number_moles, [[-36]]],
                axis=1
                )
            print(initial_log_number_moles)
            outputs = {}
            for i in range(len(CondensedT)):
                if CondensedT[i] > 0:
                    print(CondensedT[i])
                    state_gas: ThermodynamicStateProtocol = ThermodynamicState(CondensedT[i], CondensedP_bar[i])
                    mass_constraints_gas = {
                    "H": H_fraction * total_gas_mass_scaled[i],
                    "O": O_fraction * total_gas_mass_scaled[i],
                    "Si": Si_fraction * total_gas_mass_scaled[i],
                    "He": He_fraction * total_gas_mass_scaled[i],
                    "C": C_fraction * total_gas_mass_scaled[i],
                    "N": N_fraction * total_gas_mass_scaled[i],
                    }
                    #Calculating the gaseous part for H2O
                    model_atm_sol_real_condensed.solve(
                        state=state_gas,
                        mass_constraints=mass_constraints_gas,
                        initial_log_number_moles = initial_log_number_moles,
                        solver_parameters=SolverParameters(multistart=20),
                    )
                    output_atm_sol_real_condensed = model_atm_sol_real_condensed.output
                    dfs = model_atm_sol_real_condensed.output.to_dataframes()
    
                    for sheet_name, df in dfs.items():
                        df = df.copy() 
                        df["run_id"] = i
        
                        if sheet_name not in outputs:
                            outputs[sheet_name] = []
        
                        outputs[sheet_name].append(df)
                    
                    gas_dict = output_atm_sol_real_condensed.element_moles_gas()
                    Carbon_number = (gas_dict[0])[0]
                    C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
                    Hydrogen_number = (gas_dict[0])[1]
                    H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                    Helium_number = (gas_dict[0])[2]
                    He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

                    Nitrogen_number = (gas_dict[0])[3]
                    N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

                    Oxygen_number = (gas_dict[0])[4]
                    O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                    Silicon_number = (gas_dict[0])[5]
                    Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                    total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + C_mass_atm_sol_real + N_mass_atm_sol_real
                    H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                    O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                    Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                    He_fraction = He_mass_atm_sol_real/total_atm_sol_real
                    C_fraction = C_mass_atm_sol_real/total_atm_sol_real
                    N_fraction = N_mass_atm_sol_real/total_atm_sol_real
                    output_atm_sol_real_condensed.quick_look()
                    initial_log_number_moles = output_atm_sol_real_condensed.log_number_moles
    

                combined = {
                    sheet: pd.concat(df_list, ignore_index=True)
                    for sheet, df_list in outputs.items()
                    }

            # Write to Excel with SAME sheet names
            with pd.ExcelWriter(f"./excel_Uranus_time/HHeCNOSi/Precipiation/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}_hmps1_condensed_allcondensed.xlsx") as writer:
                for sheet_name, df in combined.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

[13:50:29 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, NoSolubility', 'H2_g: CombinedRealGas, NoSolubility', 'O2_g: CombinedRealGas, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility', 'C_cd: CondensateActivity, NoSolubility', 'CSi_cd: CondensateActivity, NoSolubility', 'Si_cd: CondensateActivity, NoSolubility', 'N4Si3_cd: CondensateActivity, NoSolubility', 'He_g: CombinedRealGas, NoSolubility', 'N2_g: CombinedRealGas, NoSolubility', 'H3N_g: RedlichKwong49, NoSolubility', 'CO2_g: CombinedRealGas, NoSolubility', 'CO_g: CombinedRealGas, NoSolubility', 'CH4_g: CombinedRealGas, NoSolubility')
[13:50:29 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 4000 K
[13:50:29 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2_g + 0.5 O2Si_cd = 1.0 H2O_g + 0.5 H4Si_g',
 1: '1.0 H4Si_g + 1

2510.5922005292387


[13:53:41 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:53:41 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:53:41 - atmodeller.classes             - INFO     ] - Solver steps (max) = 66
[13:53:41 - atmodeller.output_core         - DEBUG    ] - Creating Output
[13:53:41 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[13:53:41 - atmodeller.output_core         - INFO     ] - Computing asdict output
[13:53:42 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[13:53:47 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:53:47 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:53:47 - atmodeller.classes             - INFO     ] - Solver steps (max) = 52
[13:53:47 

2408.253871663796


[13:53:51 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[13:53:55 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[13:53:55 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[13:53:55 - atmodeller.classes             - INFO     ] - Solver steps (max) = 54
[13:53:55 - atmodeller.output_core         - DEBUG    ] - Creating Output
[13:53:55 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[13:53:55 - atmodeller.output_core         - INFO     ] - Computing asdict output


2310.087143781492


... (206 outputs left out)

[14:10:16 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[14:10:21 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 0 (0.00%) successful model(s)
[14:10:21 - atmodeller.classes             - WARNING  ] - 1 (100.00%) model(s) still failed


299.2334277511287


[14:10:21 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 0 attempt(s)


OverflowError: cannot convert float infinity to integer

In [35]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1


model_atm_sol_real_gas = EquilibriumModel(species_HHeCNOSi_atmosphere_gas_SiO2)
model_atm_sol_real_condensed = EquilibriumModel(species_HHeCNOSi_atmosphere_gas_SiO2_and_H2O)
model_atm_sol_real_gas_noSiO2 = EquilibriumModel(species_HHeCNOSi_atmosphere_gas_no_condensed)
rho_centre = 3000
fO2_log10_shifts = [-3]
for k in range(len(fO2_log10_shifts)):
    for i in range(len(Radii_atm)):
        if Temperatures_layer_atm.iloc[i] > 700:
            radius = Radii_atm.iloc[i]*R_earth
            masses = Masses_atm.iloc[i]*M_earth
            temp_layer = Temperatures_layer_atm.iloc[i]
            shift = f"{fO2_log10_shifts[k]:.2g}"
            hmp_text = f"{hmp:.3g}"
            file_name = f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
            P = pd.read_excel(file_name, sheet_name="state")["pressure"]
            T = pd.read_excel(file_name, sheet_name="state")["temperature"]
            All_Temperatures, All_Pressures, Tint, T_eff_end, T_atmo_1bar = Pressure_and_Temperature_to_profile(P.iloc[-1], T.iloc[-1], 1.5, 887,100)
            CondensedP, CondensedT, GasP, GasT = phase_lists(All_Temperatures, np.array(All_Pressures)*10**(-4))

            CondensedP_bar = np.array(CondensedP)*10**4
            GasP_bar = np.array(GasP)*10**4

            mask = np.array(CondensedT) > 250
            CondensedT = np.array(CondensedT)[mask]
            CondensedP_bar = CondensedP_bar[mask]
                
            atm_scalings_condensed = CondensedP_bar/max(All_Pressures)
            atm_scalings_gas = GasP_bar/max(All_Pressures)
                # Magma - Solubility - Real Gas
        
            H_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_H")[
                "gas_mass"
            ].iloc[-1]
            Si_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_Si")[
                "gas_mass"
            ].iloc[-1]
            O_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_O")[
                "gas_mass"
            ].iloc[-1]
            He_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_He")[
                "gas_mass"
            ].iloc[-1]
            C_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_C")[
                "gas_mass"
            ].iloc[-1]
            N_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_N")[
                "gas_mass"
            ].iloc[-1]


            total_gas_mass = H_mass_atm_magma_sol_real + Si_mass_atm_magma_sol_real + O_mass_atm_magma_sol_real + He_mass_atm_magma_sol_real + N_mass_atm_magma_sol_real + C_mass_atm_magma_sol_real
                
                 
            H_fraction = H_mass_atm_magma_sol_real/total_gas_mass 
            O_fraction = O_mass_atm_magma_sol_real/total_gas_mass 
            Si_fraction = Si_mass_atm_magma_sol_real/total_gas_mass
            He_fraction = He_mass_atm_magma_sol_real/total_gas_mass 
            C_fraction = C_mass_atm_magma_sol_real/total_gas_mass 
            N_fraction = N_mass_atm_magma_sol_real/total_gas_mass 
                
            total_gas_mass_scaled = atm_scalings_gas*total_gas_mass
            total_condensed_mass_scaled = atm_scalings_condensed*total_gas_mass

            state_gas: ThermodynamicStateProtocol = ThermodynamicState(GasT[0], GasP_bar[0])
            mass_constraints_gas = {
            "H": H_fraction * total_gas_mass_scaled[0],
            "O": O_fraction * total_gas_mass_scaled[0],
            "Si": Si_fraction * total_gas_mass_scaled[0],
            "He": He_fraction * total_gas_mass_scaled[0],
            "C": C_fraction * total_gas_mass_scaled[0],
            "N": N_fraction * total_gas_mass_scaled[0],
            }
            #Calculating the gaseous part for H2O
            model_atm_sol_real_gas_noSiO2.solve(
                state=state_gas,
                mass_constraints=mass_constraints_gas,
                solver_parameters=SolverParameters(multistart=200),
            )
            output_atm_sol_real_gas = model_atm_sol_real_gas_noSiO2.output
                    
            gas_dict = output_atm_sol_real_gas.element_moles_gas()

            Carbon_number = (gas_dict[0])[0]
            C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
            Hydrogen_number = (gas_dict[0])[1]
            H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

            Helium_number = (gas_dict[0])[2]
            He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

            Nitrogen_number = (gas_dict[0])[3]
            N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

            Oxygen_number = (gas_dict[0])[4]
            O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

            Silicon_number = (gas_dict[0])[5]
            Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

            total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + C_mass_atm_sol_real + N_mass_atm_sol_real
                    
            H_fraction = H_mass_atm_sol_real/total_atm_sol_real
            O_fraction = O_mass_atm_sol_real/total_atm_sol_real
            Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
            He_fraction = He_mass_atm_sol_real/total_atm_sol_real
            C_fraction = C_mass_atm_sol_real/total_atm_sol_real
            N_fraction = N_mass_atm_sol_real/total_atm_sol_real
            output_atm_sol_real_gas.quick_look()
                

#First the gaseous H2O part, then later H2O condensed
            outputs = {}
            for i in range(len(GasT)-1):
                print(GasT[i+1])
                state_gas: ThermodynamicStateProtocol = ThermodynamicState(GasT[i+1], GasP_bar[i+1])
                mass_constraints_gas = {
                "H": H_fraction * total_gas_mass_scaled[i+1],
                "O": O_fraction * total_gas_mass_scaled[i+1],
                "Si": Si_fraction * total_gas_mass_scaled[i+1],
                "He": He_fraction * total_gas_mass_scaled[i+1],
                "C": C_fraction * total_gas_mass_scaled[i+1],
                "N": N_fraction * total_gas_mass_scaled[i+1],
                }
                #Calculating the gaseous part for H2O
                model_atm_sol_real_gas.solve(
                    state=state_gas,
                    mass_constraints=mass_constraints_gas,
                    solver_parameters=SolverParameters(multistart=200),
                )
                output_atm_sol_real_gas = model_atm_sol_real_gas.output
                dfs = model_atm_sol_real_gas.output.to_dataframes()
    
                for sheet_name, df in dfs.items():
                    df = df.copy() 
                    df["run_id"] = i
        
                    if sheet_name not in outputs:
                        outputs[sheet_name] = []
        
                    outputs[sheet_name].append(df)
                    
                gas_dict = output_atm_sol_real_gas.element_moles_gas()

                Carbon_number = (gas_dict[0])[0]
                C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
                Hydrogen_number = (gas_dict[0])[1]
                H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                Helium_number = (gas_dict[0])[2]
                He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

                Nitrogen_number = (gas_dict[0])[3]
                N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

                Oxygen_number = (gas_dict[0])[4]
                O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                Silicon_number = (gas_dict[0])[5]
                Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + N_mass_atm_sol_real + C_mass_atm_sol_real
                H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                He_fraction = He_mass_atm_sol_real/total_atm_sol_real
                C_fraction = C_mass_atm_sol_real/total_atm_sol_real
                N_fraction = N_mass_atm_sol_real/total_atm_sol_real
                output_atm_sol_real_gas.quick_look()

            combined = {
                sheet: pd.concat(df_list, ignore_index=True)
                for sheet, df_list in outputs.items()
                }

            # Write to Excel with SAME sheet names
            with pd.ExcelWriter(f"./excel_Uranus_time/HHeCNOSi/Precipiation/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}_hmps1_gas_SiO2.xlsx") as writer:
                for sheet_name, df in combined.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

#Now gaseous part 
            outputs = {}
            for i in range(len(CondensedT)):
                if CondensedT[i] > 0:
                    print(CondensedT[i])
                    state_gas: ThermodynamicStateProtocol = ThermodynamicState(CondensedT[i], CondensedP_bar[i])
                    mass_constraints_gas = {
                    "H": H_fraction * total_gas_mass_scaled[i],
                    "O": O_fraction * total_gas_mass_scaled[i],
                    "Si": Si_fraction * total_gas_mass_scaled[i],
                    "He": He_fraction * total_gas_mass_scaled[i],
                    "C": C_fraction * total_gas_mass_scaled[i],
                    "N": N_fraction * total_gas_mass_scaled[i],
                    }
                    #Calculating the gaseous part for H2O
                    model_atm_sol_real_condensed.solve(
                        state=state_gas,
                        mass_constraints=mass_constraints_gas,
                        solver_parameters=SolverParameters(multistart=20),
                    )
                    output_atm_sol_real_condensed = model_atm_sol_real_condensed.output
                    dfs = model_atm_sol_real_condensed.output.to_dataframes()
    
                    for sheet_name, df in dfs.items():
                        df = df.copy() 
                        df["run_id"] = i
        
                        if sheet_name not in outputs:
                            outputs[sheet_name] = []
        
                        outputs[sheet_name].append(df)
                    
                    gas_dict = output_atm_sol_real_condensed.element_moles_gas()
                    Carbon_number = (gas_dict[0])[0]
                    C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
                    Hydrogen_number = (gas_dict[0])[1]
                    H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                    Helium_number = (gas_dict[0])[2]
                    He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

                    Nitrogen_number = (gas_dict[0])[3]
                    N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

                    Oxygen_number = (gas_dict[0])[4]
                    O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                    Silicon_number = (gas_dict[0])[5]
                    Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                    total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + C_mass_atm_sol_real + N_mass_atm_sol_real
                    H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                    O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                    Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                    He_fraction = He_mass_atm_sol_real/total_atm_sol_real
                    C_fraction = C_mass_atm_sol_real/total_atm_sol_real
                    N_fraction = N_mass_atm_sol_real/total_atm_sol_real
                    output_atm_sol_real_condensed.quick_look()
    

                combined = {
                    sheet: pd.concat(df_list, ignore_index=True)
                    for sheet, df_list in outputs.items()
                    }

            # Write to Excel with SAME sheet names
            with pd.ExcelWriter(f"./excel_Uranus_time/HHeCNOSi/Precipiation/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}_hmps1_condensed_SiO2.xlsx") as writer:
                for sheet_name, df in combined.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

[17:04:15 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, NoSolubility', 'H2_g: CombinedRealGas, NoSolubility', 'O2_g: CombinedRealGas, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'He_g: CombinedRealGas, NoSolubility', 'N2_g: CombinedRealGas, NoSolubility', 'H3N_g: RedlichKwong49, NoSolubility', 'CO2_g: CombinedRealGas, NoSolubility', 'CO_g: CombinedRealGas, NoSolubility', 'CH4_g: CombinedRealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[17:04:15 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[17:04:15 - atmodeller.classes             - INFO     ] - reactions = {0: '0.5 O2_g + 1.0 H4Si_g = 2.0 H2_g + 1.0 OSi_g',
 1: '1.5 H2_g + 0.5 N2_g = 1.0 H3N_g',
 2: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 3: '1.0 CO2_g = 0.5 O2_g + 1.0 CO_g',
 4: '2.0 H2_g + 1.0 CO2_g = 1.0 O2_g + 1.0 CH4_g',
 5: '1.0 O2_g + 1.0 H4Si_g = 

2510.5922005292387


[17:11:23 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[17:11:23 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[17:11:23 - atmodeller.classes             - INFO     ] - Solver steps (max) = 48
[17:11:23 - atmodeller.output_core         - DEBUG    ] - Creating Output
[17:11:23 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[17:11:23 - atmodeller.output_core         - INFO     ] - Computing asdict output
[17:11:29 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[17:11:39 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[17:11:39 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[17:11:39 - atmodeller.classes             - INFO     ] - Solver steps (max) = 49
[17:11:39 

2408.253871663796


[17:11:39 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[17:11:39 - atmodeller.output_core         - INFO     ] - Computing asdict output
[17:11:42 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[17:11:47 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[17:11:47 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[17:11:47 - atmodeller.classes             - INFO     ] - Solver steps (max) = 51
[17:11:47 - atmodeller.output_core         - DEBUG    ] - Creating Output
[17:11:47 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[17:11:47 - atmodeller.output_core         - INFO     ] - Computing asdict output


2310.087143781492


... (205 outputs left out)

311.8092607307661


[17:32:42 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[17:32:46 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 0 (0.00%) successful model(s)
[17:32:46 - atmodeller.classes             - WARNING  ] - 1 (100.00%) model(s) still failed
[17:32:46 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 0 attempt(s)


299.2334277511287


OverflowError: cannot convert float infinity to integer

In [31]:
init_metallicitys = [1]
init_metallicity = 1
mantle_melt_fraction= 1

model_atm_sol_real_gas = EquilibriumModel(species_HHeCNOSi_atmosphere_gas_C) 
rho_centre = 3000
fO2_log10_shifts = [-3,-2]
for k in range(len(fO2_log10_shifts)):
    for i in range(len(Radii_atm)):
        if Temperatures_layer_atm.iloc[i] > 700:
            radius = Radii_atm.iloc[i]*R_earth
            masses = Masses_atm.iloc[i]*M_earth
            temp_layer = Temperatures_layer_atm.iloc[i]
            shift = f"{fO2_log10_shifts[k]:.2g}"
            hmp_text = f"{hmp:.3g}"
            file_name = f"./excel_Uranus_time/HHeCNOSi/rho{rho_centre}/core_boundary/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}K.xlsx"
            P = pd.read_excel(file_name, sheet_name="state")["pressure"]
            T = pd.read_excel(file_name, sheet_name="state")["temperature"]
            All_Temperatures, All_Pressures, Tint, T_eff_end, T_atmo_1bar = Pressure_and_Temperature_to_profile(P.iloc[-1], T.iloc[-1], 1.5, 887,100)

            P_bar = np.array(All_Pressures)
            mask = np.array(All_Temperatures) > 250
            Temperatures = np.array(All_Temperatures)[mask]
            P_bar = P_bar[mask]

            idx_Temp = np.argsort(-Temperatures)
            
            P_bar = P_bar[idx_Temp]
            Temperatures = Temperatures[idx_Temp] 
            print(Temperatures)
            print(P_bar)
                
            atm_scalings = P_bar/max(All_Pressures)
                # Magma - Solubility - Real Gas
        
            H_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_H")[
                "gas_mass"
            ].iloc[-1]
            Si_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_Si")[
                "gas_mass"
            ].iloc[-1]
            O_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_O")[
                "gas_mass"
            ].iloc[-1]
            He_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_He")[
                "gas_mass"
            ].iloc[-1]
            C_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_C")[
                "gas_mass"
            ].iloc[-1]
            N_mass_atm_magma_sol_real = pd.read_excel(file_name, sheet_name="element_N")[
                "gas_mass"
            ].iloc[-1]


            total_gas_mass = H_mass_atm_magma_sol_real + Si_mass_atm_magma_sol_real + O_mass_atm_magma_sol_real + He_mass_atm_magma_sol_real + N_mass_atm_magma_sol_real + C_mass_atm_magma_sol_real
                
                 
            H_fraction = H_mass_atm_magma_sol_real/total_gas_mass 
            O_fraction = O_mass_atm_magma_sol_real/total_gas_mass 
            Si_fraction = Si_mass_atm_magma_sol_real/total_gas_mass
            He_fraction = He_mass_atm_magma_sol_real/total_gas_mass 
            C_fraction = C_mass_atm_magma_sol_real/total_gas_mass 
            N_fraction = N_mass_atm_magma_sol_real/total_gas_mass 
                
            total_gas_mass_scaled = atm_scalings*total_gas_mass
                

#First the gaseous H2O part, then later H2O condensed
            outputs = {}
            for i in range(len(Temperatures)):
                print(Temperatures[i])
                state_gas: ThermodynamicStateProtocol = ThermodynamicState(Temperatures[i], P_bar[i])
                mass_constraints_gas = {
                "H": H_fraction * total_gas_mass_scaled[i],
                "O": O_fraction * total_gas_mass_scaled[i],
                "Si": Si_fraction * total_gas_mass_scaled[i],
                "He": He_fraction * total_gas_mass_scaled[i],
                "C": C_fraction * total_gas_mass_scaled[i],
                "N": N_fraction * total_gas_mass_scaled[i],
                }
                #Calculating the gaseous part for H2O
                model_atm_sol_real_gas.solve(
                    state=state_gas,
                    mass_constraints=mass_constraints_gas,
                    solver_parameters=SolverParameters(multistart=200),
                )
                output_atm_sol_real_gas = model_atm_sol_real_gas.output
                dfs = model_atm_sol_real_gas.output.to_dataframes()
    
                for sheet_name, df in dfs.items():
                    df = df.copy() 
                    df["run_id"] = i
        
                    if sheet_name not in outputs:
                        outputs[sheet_name] = []
        
                    outputs[sheet_name].append(df)
                    
                gas_dict = output_atm_sol_real_gas.element_moles_gas()

                Carbon_number = (gas_dict[0])[0]
                C_mass_atm_sol_real = 12.01*Carbon_number*10**(-3)
                    
                Hydrogen_number = (gas_dict[0])[1]
                H_mass_atm_sol_real = 1.008*Hydrogen_number*10**(-3)

                Helium_number = (gas_dict[0])[2]
                He_mass_atm_sol_real = 4.00*Helium_number*10**(-3)

                Nitrogen_number = (gas_dict[0])[3]
                N_mass_atm_sol_real = 14.01*Nitrogen_number*10**(-3)

                Oxygen_number = (gas_dict[0])[4]
                O_mass_atm_sol_real = 16.00*Oxygen_number*10**(-3)

                Silicon_number = (gas_dict[0])[5]
                Si_mass_atm_sol_real = 28.085*Silicon_number*10**(-3)

                total_atm_sol_real = H_mass_atm_sol_real + O_mass_atm_sol_real + Si_mass_atm_sol_real + He_mass_atm_sol_real + N_mass_atm_sol_real + C_mass_atm_sol_real
                H_fraction = H_mass_atm_sol_real/total_atm_sol_real
                O_fraction = O_mass_atm_sol_real/total_atm_sol_real
                Si_fraction = Si_mass_atm_sol_real/total_atm_sol_real
                He_fraction = He_mass_atm_sol_real/total_atm_sol_real
                C_fraction = C_mass_atm_sol_real/total_atm_sol_real
                N_fraction = N_mass_atm_sol_real/total_atm_sol_real
                output_atm_sol_real_gas.quick_look()

            combined = {
                sheet: pd.concat(df_list, ignore_index=True)
                for sheet, df_list in outputs.items()
                }

            # Write to Excel with SAME sheet names
            with pd.ExcelWriter(f"./excel_Uranus_time/HHeCNOSi/Precipiation/HHeCNOSi_magma_sol_real_{init_metallicity}xsolar_melt{round(mantle_melt_fraction * 100)}_fO2{shift}_T{temp_layer:.4g}_hmps1_C_cd.xlsx") as writer:
                for sheet_name, df in combined.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)

[14:43:34 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: CombinedRealGas, NoSolubility', 'H2_g: CombinedRealGas, NoSolubility', 'O2_g: CombinedRealGas, NoSolubility', 'OSi_g: CombinedRealGas, NoSolubility', 'H4Si_g: CombinedRealGas, NoSolubility', 'He_g: CombinedRealGas, NoSolubility', 'N2_g: CombinedRealGas, NoSolubility', 'H3N_g: RedlichKwong49, NoSolubility', 'CO2_g: CombinedRealGas, NoSolubility', 'CO_g: CombinedRealGas, NoSolubility', 'CH4_g: CombinedRealGas, NoSolubility', 'C_cd: CondensateActivity, NoSolubility')
[14:43:34 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[14:43:34 - atmodeller.classes             - INFO     ] - reactions = {0: '0.5 O2_g + 1.0 H4Si_g = 2.0 H2_g + 1.0 OSi_g',
 1: '1.5 H2_g + 0.5 N2_g = 1.0 H3N_g',
 2: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 3: '1.0 CO2_g = 0.5 O2_g + 1.0 CO_g',
 4: '2.0 H2_g + 1.0 CO2_g = 1.0 O2_g + 1.0 CH4_g',
 5: '1.0 CO2_g = 1.0 O2_g + 1.0 

[2617.2793999999835  2510.5922005292387  2408.253871663796
 2310.087143781492   2215.921973507591   2125.595249188613
 2038.950508376504   1955.8376668341705  1876.1127585934544
 1799.6376866158114  1726.279983624398   1655.9125826939728
 1588.4135972020124  1523.6661097607653  1461.557969765635
 1401.9815992103536  1344.8338064338634  1290.0156074777547
 1237.4320547464642  1186.992072675329   1138.6083001239617
 1092.1969392243575  1047.677610424636   1004.973213480419
  964.0097941565518   924.716416412237    887.0250398526592
  850.8704022399071   816.1899068654138   782.9235145953254
  751.0136404091394   720.4050542607038   691.0447860992278
  662.8820348963762   635.8680815338174   609.9562054138148
  585.101604663604    561.2613198124322   538.394160828281
  516.4606374094599   495.42289243450824  475.244638482163
  455.8910973415939   437.32894244165584  419.52624413657736
  402.4524177942581   386.07817464214224  370.37547533437413
  355.31748621248823  340.8785382400178   32

[14:45:49 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[14:45:49 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[14:45:49 - atmodeller.classes             - INFO     ] - Solver steps (max) = 39
[14:45:49 - atmodeller.output_core         - DEBUG    ] - Creating Output
[14:45:49 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[14:45:49 - atmodeller.output_core         - INFO     ] - Computing asdict output
[14:45:51 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[14:45:57 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[14:45:57 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[14:45:57 - atmodeller.classes             - INFO     ] - Solver steps (max) = 39
[14:45:57 

2510.5922005292387


[14:45:59 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[14:46:05 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 1 (100.00%) successful model(s)
[14:46:05 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 1 attempt(s)
[14:46:05 - atmodeller.classes             - INFO     ] - Solver steps (max) = 39
[14:46:05 - atmodeller.output_core         - DEBUG    ] - Creating Output
[14:46:05 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[14:46:05 - atmodeller.output_core         - INFO     ] - Computing asdict output


2408.253871663796


... (212 outputs left out)

[15:01:32 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2


299.2334277511287


[15:01:36 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 0 (0.00%) successful model(s)
[15:01:36 - atmodeller.classes             - WARNING  ] - 1 (100.00%) model(s) still failed
[15:01:36 - atmodeller.classes             - INFO     ] - Multistart summary: 1 (100.00%) models(s) required 0 attempt(s)


OverflowError: cannot convert float infinity to integer